In [89]:
import pandas as pd
import sys
import numpy as np
import warnings

sys.path.append("/Users/ejowik001/Desktop/Github/Nowcasting/kedro/refinery/dependencies/")

In [90]:
from estimation import auto_train_evaluate, cast_to_base_unit
from plots import plot_prediction
from retransform_prediction import retransform_
from retransform_data import retransform_data
from utils import cast_spec_to_dict, _convert_to_datetime

ImportError: cannot import name 'cast_to_base_unit' from 'estimation' (/Users/ejowik001/Desktop/Github/Nowcasting/kedro/refinery/dependencies/estimation.py)

In [91]:
EPSILON = 1e-10

def rrse(actual: np.ndarray, predicted: np.ndarray, benchmark: np.ndarray=None):
    """ Root Relative Squared Error """
    return np.sqrt(
        np.sum(np.square(actual - predicted))
        / np.sum(np.square(actual - benchmark))
    )

def _error(actual: np.ndarray, predicted: np.ndarray):
    """ Simple error """
    return actual - predicted

def _percentage_error(actual: np.ndarray, predicted: np.ndarray):
    """
    Percentage error

    Note: result is NOT multiplied by 100
    """
    return _error(actual, predicted) / (actual + EPSILON)

def mape(actual: np.ndarray, predicted: np.ndarray):
    """
    Mean Absolute Percentage Error

    Note: result is NOT multiplied by 100
    """
    return np.mean(np.abs(_percentage_error(actual, predicted)))

def mse(actual: np.ndarray, predicted: np.ndarray):
    """ Mean Squared Error """
    return np.mean(np.square(_error(actual, predicted)))


def rmse(actual: np.ndarray, predicted: np.ndarray):
    """ Root Mean Squared Error """
    return np.sqrt(mse(actual, predicted))


In [92]:
def assign_weights(s):
    if (s['directional_accuracy'] == -1) and (s['within_cbounds'] == -1):
        return 2
    elif (s['directional_accuracy'] == -1) and (s['within_cbounds'] == 1):
        return 1.75
    elif (s['directional_accuracy'] == 1) and (s['within_cbounds'] == -1):
        return 1.25
    elif (s['directional_accuracy'] == 1) and (s['within_cbounds'] == 1):
        return 1
    else: return np.infty

In [93]:
confidence_bounds_func = lambda row: row['lower']<=row['y_pred']<=row['upper']

In [94]:
def wdmpe(predicted, actual):
    actual_diff = actual.sort_index().diff()
    actual_signs = np.sign(actual_diff)
    predicted_diff = predicted.sort_index().diff()
    predicted_signs = np.sign(predicted_diff)

    resid = predicted-actual

    dir_acc = list(actual_signs * predicted_signs)

    resid_mean = resid.expanding(1).mean()
    resid_std = resid.expanding(2).std().fillna(0)

    lower = actual-resid_std
    upper = actual+resid_std

    df = pd.DataFrame({
        "directional_accuracy": dir_acc,
        "lower": lower,
        "upper": upper,
        "y_pred": predicted
    }).iloc[1:, :]
    df['within_cbounds'] = df.apply(confidence_bounds_func, axis=1).astype(int).replace({0: -1})
    df['percentage_error'] = resid / actual

    df['weights'] = df.apply(assign_weights, axis=1)
    df["weighted_percentage_error"] = df['weights'] * df['percentage_error']
    return df["weighted_percentage_error"].mean()


In [95]:
# def cast_to_base_unit(ds, model_result, spec, series_name):
#     Spec = cast_spec_to_dict(spec.loc[spec["seriesid"] == series_name])

#     ## Retransform
#     ds = _convert_to_datetime(ds, ['ReferenceDate'])

#     dsrc = ds.set_index('ReferenceDate')

#     # def retransform_prediction(transf_series, base_series, Spec, series_name):
#     base_series = dsrc[series_name]
#     header = [series_name]

#     backcast = model_result['predictions']['backcast']
#     forecast = pd.Series(
#         model_result["predictions"]["forecast"],
#         index=[model_result["predictions"]["reference_date"]]
#         )

#     transf_pred = pd.concat([backcast, forecast])
#     transf_pred.index = pd.to_datetime(transf_pred.index)

#     transf_series = model_result["actual"]

#     Time = np.sort(np.unique(np.concatenate((base_series.index.date, transf_pred.index.date))))
#     cutoff_date = transf_pred.index.min().date()

#     Z = base_series.reindex(Time).to_numpy().reshape(-1,1)

#     Yhat = transf_pred.reindex(Time).to_numpy().reshape(-1,1)
#     Y = transf_series.reindex(Time).to_numpy().reshape(-1,1)

#     Rhat = retransform_(X=Yhat, Z=Z, Time=Time, Spec=Spec, header=header, cutoff_date=cutoff_date)
#     R = retransform_data(X=Y, Z=Z, Time=Time, Spec=Spec, header=header, cutoff_date=cutoff_date)

#     return Rhat, R, Time, cutoff_date

In [96]:
series_name = "PCEC96"
reference_date = "2024-08-01"
n_periods = 60

In [97]:
ds = pd.read_parquet("../data/04_feature/selected_series.parquet")


In [98]:
# Example usage
best_model_result = auto_train_evaluate(
    ds=ds,
    ref_date_col="ReferenceDate",
    series_name=series_name,
    reference_date=reference_date,
    n_periods=n_periods,
)

# Print the best model's details
print(f"Best Model: {best_model_result['best_model']}")
print(f"R-Squared: {best_model_result['r_squared']}")
print(f"MAPE: {best_model_result['mape']}")
print(f"RMSE: {best_model_result['rmse']}")
print(f"Forecast: {best_model_result['predictions']['forecast']}")

Time = best_model_result['predictions']['backcast'].index
plot_prediction(dt=Time, y_pred=best_model_result['predictions']['backcast'], y_actual=best_model_result['actual'].loc[Time], mode="lines+markers")

Best Model: LinearForest
R-Squared: 0.8891242070474648
MAPE: 3.85947842233644
RMSE: 0.9157592908371768
Forecast: 0.0149648692582693


In [ ]:
ds_spec = pd.read_csv("../data/02_intermediate/variable.csv")
ds = pd.read_parquet("../data/02_intermediate/non_transformed_data.parquet")

Rhat, R, Time, cutoff_date = cast_to_base_unit(ds, best_model_result, ds_spec, series_name)
header = [series_name]
Rhat_df = pd.DataFrame(Rhat, columns=header, index=Time)
R_df = pd.DataFrame(R, columns=header, index=Time)

Z_df = pd.DataFrame(ds[series_name], columns=header, index=Time)

dt = Time
plot_prediction(dt=dt, y_pred=Rhat_df.loc[dt][series_name], y_actual=R_df.loc[dt][series_name], mode="lines+markers")

# print(f"y_pred: {R_df.loc[dt][series_name].item()}, y_actual: {Z_df.loc[dt][series_name].item()}")